In [4]:
import json, os

In [29]:
folder = "/home/lamdo/IllinoisRetrievalBenchmark/test_run_with_gpt_gitig_"

In [30]:
url_content_path = os.path.join(folder, "step2_1")
modified_facts_path = os.path.join(folder, "step2_2")
groundedness_checking_only_path = os.path.join(folder, "step3")
groundedness_checking_with_correction_path = os.path.join(folder, "step3_with_correction")

In [34]:
files = os.listdir(groundedness_checking_with_correction_path)


gc_num_pos = 0
gcwc_num_pos = 0
fixed_facts = {}
for file in files:
    gc_fullpath = os.path.join(groundedness_checking_only_path, file)
    mf_fullpath = os.path.join(modified_facts_path, file)
    uc_fullpath = os.path.join(url_content_path, file)
    gcwc_fullpath = os.path.join(groundedness_checking_with_correction_path, file)

    with open(gc_fullpath) as f:
        gc_meta = json.load(f)
        gc_data = gc_meta.get("groundedness_check")
    
    with open(mf_fullpath) as f:
        mf_meta = json.load(f)
        mf_data = {int(k):v for k,v in  mf_meta.get("modified_fact_mapper").items()}

    with open(uc_fullpath) as f:
        uc_meta = json.load(f)
        uc_data = uc_meta.get("url_content_mapper")
    
    with open(gcwc_fullpath) as f:
        gcwc_meta = json.load(f)
        gcwc_data = gcwc_meta.get("groundedness_check")
        gcwc_fc = {int(k): v for k,v in gcwc_meta.get("fact_correction").items()}

    fixed_facts[gc_meta["title"]] = []

    assert len(gc_data) == len(gcwc_data)

    gc_num_pos += len({k: v for k,v in gc_data.items() if v})
    gcwc_num_pos += len({k: v for k,v in gcwc_data.items() if v})

    for k in gcwc_data:
        if gcwc_data[k] == 1 and gc_data[k] == 0: 
            fact_id = int(k.split("--__--")[0])
            url = k.split("--__--")[1]
            fixed_facts[gc_meta["title"]].append(
                {"before": mf_data[fact_id], "after": gcwc_fc[fact_id], "url_content": uc_data[url]}
            )

In [35]:
gc_num_pos, gcwc_num_pos

(45, 96)

In [37]:
fixed_facts

{'Acrimony (band)': [],
 'Alexandre Rousselet': [],
 'Atocha station memorial': [],
 'Barton Academy (Vermont)': [],
 'Berney Arms': [{'before': 'Ashtree Farm in Berney Arms is used by the RSPB as a series of dwellings and as its base for the marshes.',
   'after': 'Ashtree Farmhouse in Berney Arms has been extended and turned into three dwellings by the RSPB.',
   'url_content': {'url': 'https://web.archive.org/web/20140223140508/http://www.broads-authority.gov.uk/broads/live/planning/landscape-character-assessment/Area_19_-_Halvergate_Marshes.pdf',
    'accessible': True,
    'url_content': "Broads Landscape Character Assessment (2006)\n1\nLocal Character Area 19. Halvergate Marshes (excluding Bure Loop and the west of Tunstall Dyke)\n1. Part of the Halvergate Marshes from the air (Pho to: Mike Page)\nSurvey Points : TG 44470664; TG 43810480; TG 42270925\nBoundaries\nThis area is bounded by the river Bure to the north, the river\nWaveney and the northern bank of Breydon to the south 